# 第十七课｜外部内存是什么？

片上**随机存取存储器（Random-Access Memory, RAM）**靠近逻辑，但容量有限。真实**连接组（connectome）**的 synapse store 很快会超过片上容量，于是需要：

> **把大量数据放到现场可编程门阵列（Field-Programmable Gate Array, FPGA）芯片外的内存。**

主要新概念：**双倍数据速率同步动态随机存储器（Double Data Rate Synchronous Dynamic Random-Access Memory, DDR SDRAM）有独立访问延迟，连续 burst 与零散 random access 的成本不同。**

## 1. 概念账本

**已经知道：** memory hierarchy、latency、bandwidth、synapse records。

**今天学习：**
- **双倍数据速率同步动态随机存储器（Double Data Rate Synchronous Dynamic Random-Access Memory, DDR SDRAM）**；
- **word**：本课把一个可单独寻址的数据单元简称为 word；
- **burst**：把多个相邻 word 组织成一段连续批量传输，让固定启动成本由多份数据共同分担；
- sequential access 与 random access。

**只预告：** 下一课再学习访问这类内存时常见的标准片上通信协议。

## 2. 为什么不把所有 synapse 都放片上？

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 630 115" role="img" aria-label="l17-flow" style="width:100%;height:auto"><defs><marker id="l17-flow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 Z" fill="#506070"/></marker></defs><rect x="10" y="25" width="180" height="65" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="100" y="63" text-anchor="middle" font-family="sans-serif" font-size="14">synapse engine</text><path d="M190,57 h25" fill="none" stroke="#506070" marker-end="url(#l17-flow)"/><rect x="220" y="25" width="180" height="65" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="310" y="63" text-anchor="middle" font-family="sans-serif" font-size="14">memory controller</text><path d="M400,57 h25" fill="none" stroke="#506070" marker-end="url(#l17-flow)"/><rect x="430" y="25" width="180" height="65" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="520" y="63" text-anchor="middle" font-family="sans-serif" font-size="14">external DDR</text></svg>

外部 DDR 提供更大容量，但访问路径更长、控制更复杂。本项目使用平台已经提供的 memory controller，不要求学生自己实现 DDR 的底层物理接口电路。

## 3. 为什么连续访问通常更友好？

很多访问存在固定启动成本。相邻地址若能合并成 burst，这个成本可以由多份数据共同分担；random access 更容易频繁重新开始一次新的请求/传输。

## 4. 一个教学 burst model

把地址看成 word index。只有当 `current == previous + 1` 且当前 burst 未满时，才继续同一个 burst。这个模型只观察 access pattern，不是 DDR controller。

## 5. Run：连续地址与 random-like 地址

先预测 8 个连续 word、最大 burst 长度 4 时需要几个 burst。

### 运行前：word 地址与字节地址

本模型使用 word 下标，每次 `address+1` 访问下一个数据单元。若一个 word 是 4 bytes，其字节地址依次为 0、4、8、12，不能把字节地址的 `+1` 当成下一个完整 word。

地址 0 到 7、最长 4-word burst 分成 `[0,1,2,3]` 和 `[4,5,6,7]`；`[0,9,2]` 分成三个单独段。DDR 的“双倍数据速率”指物理数据在时钟两种边沿上传送；本例估算的是逻辑请求分组，真实控制器还需处理固定 burst、行地址、刷新和调度。


In [ ]:
def estimate_bursts(addresses, max_burst_words):
    if not addresses:
        return 0
    bursts = 1
    run_length = 1
    for previous, current in zip(addresses, addresses[1:]):
        if current == previous + 1 and run_length < max_burst_words:
            run_length += 1
        else:
            bursts += 1
            run_length = 1
    return bursts

sequential = list(range(8))
random_like = [0, 9, 2, 14, 7, 20, 1, 30]

print("sequential bursts:", estimate_bursts(sequential, 4))
print("random-like bursts:", estimate_bursts(random_like, 4))


## 6. Observe

8 个连续地址组成两个 4-word burst；random-like 地址几乎每个都需要新 burst。总 bytes 相同，访问模式仍会改变有效成本。

## 7. integrity test 为什么先于性能？

DDR hello-world 先写入一批已知值，再可靠读回。只有 read/write integrity 稳定后，benchmark bandwidth 才有意义。

## 8. Try It

把 sequential 改成 `list(range(10))`，max burst 仍为 4；再把 max burst 改成 8。分别先预测 burst 数。

## 9. 作业

[第 17 课作业：估算 sequential/random access 的 burst 成本](../../exercises/zh/17_external_memory_ddr.ipynb)

## 10. AI Task

让 AI 解释“同样 1 KB 数据，连续读取与随机读取为什么可能不同”。检查它不要误说 random access 不能工作；这里讨论的是效率。

## 11. Human Check

解释 DDR 为什么解决容量但引入访问成本；burst 为什么适合连续数据；random/sequential 为什么影响有效 bandwidth；为什么课程使用平台提供的 memory controller，而不是让学生自己实现 DDR 的底层物理接口。

## 12. Engineering Handoff

对应 `RMD-014`：通过平台提供的 memory controller 做稳定 read/write + integrity test。

## 13. 项目追踪 Project Trace

- Lesson: `LSN-017`
- Mapping: `RMD-014`
- First proof: DDR read/write integrity
- Performance preview: sequential vs random vs burst

## 14. Exit Ticket

你能解释外部 DDR 为什么容量大但访问成本不同，并能根据简单地址序列判断 burst 数。